In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')
# データフレームを表示して確認
df.head()

,カテゴリー,商品コード,商品名,売上日,単価,数量,原価
0,食品,1001,りんご,2023-01-01,200,50,120
1,食品,1002,バナナ,2023-01-01,150,100,80
2,食品,1003,牛乳,2023-01-02,180,80,100
3,衣服,2001,Tシャツ,2023-01-02,1500,20,800
4,衣服,2002,ジーンズ,2023-01-03,5000,10,2500


In [3]:
# 2. データをLLM用にテキスト形式に変換
# データフレーム全体を文字列に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"
# 表示して確認
print(prompt_text)

売上データ:
    カテゴリー 商品コード      商品名         売上日    単価   数量    原価
0      食品  1001      りんご  2023-01-01   200   50   120
1      食品  1002      バナナ  2023-01-01   150  100    80
2      食品  1003       牛乳  2023-01-02   180   80   100
3      衣服  2001     Tシャツ  2023-01-02  1500   20   800
4      衣服  2002     ジーンズ  2023-01-03  5000   10  2500
..    ...   ...      ...         ...   ...  ...   ...
235    衣服  2077   レインパンツ  2023-04-28  2000   18  1000
236    食品  1085      ザクロ  2023-04-29   600   40   300
237   日用品  3077    バスブラシ  2023-04-29   400   60   200
238    衣服  2078  レインシューズ  2023-04-30  2500   15  1250
239    食品  1086    ココナッツ  2023-04-30   300   80   150

[240 rows x 7 columns]
この売上データの傾向を分析してください。


In [4]:
# 3. OpenAI APIの呼び出し

# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

この売上データからいくつかの傾向を分析するため、以下のポイントに注目してみましょう：

### 1. 売上高の合計とカテゴリー別の分析
各カテゴリーの総売上、販売数量、原価を分析します。このデータを用いて、どのカテゴリーが最も利益を生んでいるかを確認します。

### 2. 利益率の計算
各商品の利益を計算し、その利益率を分析することで、どの商品が効率的に利益を生み出しているかを把握します。利益は以下の式で計算できます：
\[ 
\text{利益} = \text{売上} - \text{原価} 
\]
利益率は以下の式で計算できます：
\[ 
\text{利益率} = \left( \frac{\text{利益}}{\text{売上}} \right) \times 100 
\]

### 3. 時間的なトレンド
売上データを日別、月別に集計し、時間とともにどのように売上が推移しているかを確認します。特定の時期に特に売上が伸びたのか、あるいは停滞しているのかを把握します。

### 4. 商品コードや商品名によるパフォーマンス比較
各商品ごとに売上高、販売数量、利益を比較し、どの商品のパフォーマンスが良いかを特定します。また、人気商品と不人気商品を把握します。

### 5. 在庫管理と需要予測
売上データを基に在庫回転率を計算し、需要予測を行うことで、在庫管理に役立てることができます。特に、売れ筋商品については適切な在庫量を確保する必要があります。

### 分析結果の仮定
具体的な分析結果はデータに基づくため、例えば以下の仮定が成り立つかもしれません：

- **食品カテゴリーが全体の売上の大半を占めている**場合、今後もこのカテゴリーへの注力が必要。
- **衣服カテゴリーは利益率が高い**かもしれないが、売上量が少ない場合、マーケティング施策の強化が検討されるべき。
- **季節需要**が強い商品（例：レインウェアなど）は、売上が特定の時期に集中すると予測される。

### 提案
1. **人気商品のプロモーション戦略**を強化し、需要をさらに伸ばす施策を講じる。
2. **売れていない商品の見直し**を行い、原価管理の最適化を図る。
3. **リアルタイムの売上分析システム**を導入し、変化に迅速に対応できる体制を整える。

このように、売上データを基

In [5]:
# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])
print(df_out)

                                                   結果
0          この売上データからいくつかの傾向を分析するため、以下のポイントに注目してみましょう：
1                                                    
2                             ### 1. 売上高の合計とカテゴリー別の分析
3   各カテゴリーの総売上、販売数量、原価を分析します。このデータを用いて、どのカテゴリーが最も利...
4                                                    
5                                       ### 2. 利益率の計算
6   各商品の利益を計算し、その利益率を分析することで、どの商品が効率的に利益を生み出しているかを...
7                                                 \[ 
8                  \text{利益} = \text{売上} - \text{原価} 
9                                                  \]
10                                   利益率は以下の式で計算できます：
11                                                \[ 
12  \text{利益率} = \left( \frac{\text{利益}}{\text{売上}...
13                                                 \]
14                                                   
15                                    ### 3. 時間的なトレンド
16  売上データを日別、月別に集計し、時間とともにどのように売上が推移しているかを確認します。特定...
17                          

In [6]:
# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)